# 🧠 Phase 1 — Part 6: Train CTC Models (Wav2Vec2 / WavLM / HuBERT / Data2Vec)

---

## What this notebook does
Trains **Family A (Self-Supervised) models** using CTC loss. All 4 models use the **same training script** — you only change the `MODEL_CHOICE` variable.

| Model | HuggingFace ID | When to use |
|-------|---------------|-------------|
| Wav2Vec2-XLSR-53 | `facebook/wav2vec2-large-xlsr-53` | **Must** — best self-supervised baseline |
| WavLM-Large | `microsoft/wavlm-large` | **Must** — best for noisy clinic audio |
| HuBERT-Large | `facebook/hubert-large-ls960-ft` | Recommended — strong dialect learning |
| Data2Vec-Audio | `facebook/data2vec-audio-large-960h` | Optional — compare with others |

## CTC vs Seq2Seq (How this differs from Whisper)
- **Whisper** (Part 5) is Seq2Seq: encoder-decoder, generates text token by token
- **These models** use CTC: encoder-only, predicts one character per audio frame
- CTC needs a **custom vocabulary** built from your transcripts (Whisper has its own tokenizer)

## 🔒 Same checkpoint strategy as Part 5:
- Checkpoints saved to Drive every epoch
- `resume_from_checkpoint=True` for auto-resume
- Best model (lowest WER) kept at end

---

## Step 0: Mount Drive, Load Config, Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os
config = json.load(open('/content/drive/MyDrive/NSU_499A/config.json'))
BASE = config['base_path']
print(f'✅ Project root: {BASE}')

import torch
assert torch.cuda.is_available(), '❌ NO GPU! Go to Runtime → Change runtime type → T4 GPU'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate
print('✅ Packages installed')

## Step 1: Choose Model and Dialect

Change `MODEL_CHOICE` to train a different model. Run the **entire notebook** for each model/dialect combination.

In [ ]:
# ════════════════════════════════════════════════════════════════
# CONFIGURATION — Change these for each training run
# ════════════════════════════════════════════════════════════════

MODEL_CHOICE = 'wav2vec2'   # Options: 'wav2vec2', 'wavlm', 'hubert', 'data2vec'
DIALECT      = 'dhaka'     # Options: 'dhaka', 'chittagong'

# ── Model ID mapping ──
MODEL_MAP = {
    'wav2vec2': ('facebook/wav2vec2-large-xlsr-53', 'wav2vec2-xlsr'),
    'wavlm':    ('microsoft/wavlm-large', 'wavlm-large'),
    'hubert':   ('facebook/hubert-large-ls960-ft', 'hubert-large'),
    'data2vec': ('facebook/data2vec-audio-large-960h', 'data2vec-audio'),
}

MODEL_ID, MODEL_SHORT = MODEL_MAP[MODEL_CHOICE]
OUT_DIR = os.path.join(BASE, f'models/{MODEL_SHORT}-{DIALECT}')

print(f'🎯 Model:   {MODEL_ID}')
print(f'🗣️ Dialect: {DIALECT}')
print(f'💾 Output:  {OUT_DIR}')

# Check for existing checkpoints
if os.path.exists(OUT_DIR):
    ckpts = [d for d in os.listdir(OUT_DIR) if d.startswith('checkpoint-')]
    if ckpts:
        print(f'\n🔄 Found {len(ckpts)} existing checkpoints — will RESUME')

## Step 2: Build Vocabulary from Training Transcripts

### Why is this needed?
CTC models predict **one character at a time** (not tokens like Whisper). So we need a character vocabulary built from YOUR training data. This includes all Bangla characters, spaces, and special tokens.

### What gets saved:
- `vocab.json` → inside the model output directory (permanent in Drive)

In [ ]:
import pandas as pd
import json

# Load training transcripts
train_csv = os.path.join(BASE, f'dataset/{DIALECT}_train.csv')
df = pd.read_csv(train_csv)
print(f'📊 Training samples: {len(df)}')

# Extract all unique characters
all_text = ' '.join(df.transcript.astype(str).tolist())
chars = sorted(set(all_text))

# Build vocabulary
vocab = chars + ['[UNK]', '[PAD]']
vocab_dict = {char: idx for idx, char in enumerate(vocab)}

# Save to model directory
os.makedirs(OUT_DIR, exist_ok=True)
vocab_path = os.path.join(OUT_DIR, 'vocab.json')
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)

print(f'✅ Vocabulary built: {len(vocab_dict)} characters')
print(f'💾 Saved to: {vocab_path}')
print(f'   Sample characters: {chars[:20]}')

## Step 3: Load Tokenizer, Feature Extractor, Processor, and Model

### What are these components?
- **Tokenizer**: Converts text ↔ character IDs using your vocab.json
- **Feature Extractor**: Converts raw audio → normalized waveform input
- **Processor**: Combines tokenizer + feature extractor in one object
- **Model**: The neural network with a CTC head (character prediction layer)

### `freeze_feature_encoder()`:
- Freezes the CNN feature extractor layers (don't train them)
- Only trains the Transformer layers and CTC head
- Prevents catastrophic forgetting of pre-trained features

In [ ]:
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC
)

# Tokenizer (from our vocab)
tokenizer = Wav2Vec2CTCTokenizer(
    vocab_path,
    unk_token='[UNK]',
    pad_token='[PAD]',
    word_delimiter_token=' '
)

# Feature extractor (from pretrained model)
feat_ext = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)

# Combined processor
processor = Wav2Vec2Processor(feature_extractor=feat_ext, tokenizer=tokenizer)

# Model with CTC head
print(f'🔄 Loading {MODEL_ID}...')
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_ID,
    ctc_loss_reduction='mean',
    pad_token_id=tokenizer.pad_token_id,
    vocab_size=len(vocab_dict),
    ignore_mismatched_sizes=True  # CTC head size differs from pretrained
)

# Freeze feature encoder (CNN layers)
model.freeze_feature_encoder()

print(f'✅ Model loaded: {MODEL_ID}')
print(f'   Parameters: {model.num_parameters() / 1e6:.0f}M')
print(f'   Vocab size: {len(vocab_dict)}')

## Step 4: Load and Prepare Dataset

In [ ]:
from datasets import Dataset, Audio

val_csv = os.path.join(BASE, f'dataset/{DIALECT}_val.csv')
train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
train_ds = train_ds.cast_column('file_path', Audio(sampling_rate=16000))
val_ds   = val_ds.cast_column('file_path', Audio(sampling_rate=16000))

def prepare_ctc_batch(batch):
    """Convert audio → input values and text → character IDs for CTC."""
    audio = batch['file_path']['array']
    batch['input_values'] = processor(
        audio, sampling_rate=16000
    ).input_values[0]
    batch['labels'] = tokenizer(batch['transcript']).input_ids
    return batch

print('🔄 Processing training data...')
train_ds = train_ds.map(prepare_ctc_batch, remove_columns=train_ds.column_names)
print('🔄 Processing validation data...')
val_ds = val_ds.map(prepare_ctc_batch, remove_columns=val_ds.column_names)
print(f'✅ Train: {len(train_ds)} | Val: {len(val_ds)}')

## Step 5: CTC Data Collator

In [ ]:
import dataclasses
from typing import Any, Dict, List, Union

@dataclasses.dataclass
class CTCCollator:
    processor: Any
    padding: bool = True

    def __call__(self, features):
        inp = [{'input_values': f['input_values']} for f in features]
        lbl = [{'input_ids': f['labels']} for f in features]
        
        batch = self.processor.pad(inp, padding=True, return_tensors='pt')
        labels = self.processor.pad(labels=lbl, padding=True, return_tensors='pt')
        
        batch['labels'] = labels.input_ids.masked_fill(
            labels.input_ids == self.processor.tokenizer.pad_token_id, -100
        )
        return batch

collator = CTCCollator(processor=processor)
print('✅ CTC collator ready')

## Step 6: WER Metric for CTC

In [ ]:
import evaluate
import numpy as np

wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred_str = processor.batch_decode(pred_ids)
    
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    label_str = [processor.decode(l, group_tokens=False) for l in label_ids]
    
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': round(wer, 4)}

print('✅ WER metric ready')

## Step 7: Training Arguments

### Key differences from Whisper training:
- **Higher learning rate** (`1e-4` vs `1e-5`) — CTC models benefit from faster learning
- **More epochs** (30 vs 15) — CTC takes longer to converge
- **No `predict_with_generate`** — CTC uses direct prediction, not sequence generation

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir=OUT_DIR,                        # 💾 GOOGLE DRIVE!
    
    # ── Training ──
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    num_train_epochs=30,
    
    # ── Memory ──
    fp16=torch.cuda.is_available(),
    
    # ── 🔒 CHECKPOINT SAVING ──
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    
    # ── Logging ──
    logging_steps=25,
    logging_dir=os.path.join(OUT_DIR, 'logs'),
    report_to='none',
    run_name=f'{MODEL_SHORT}-{DIALECT}',
)

print(f'✅ Training config: {args.num_train_epochs} epochs, lr={args.learning_rate}')

## Step 8: Train! (with auto-resume)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

# ── Auto-resume logic ──
resume_ckpt = None
if os.path.exists(OUT_DIR):
    ckpts = [d for d in os.listdir(OUT_DIR) if d.startswith('checkpoint-')]
    if ckpts:
        latest = sorted(ckpts, key=lambda x: int(x.split('-')[1]))[-1]
        resume_ckpt = os.path.join(OUT_DIR, latest)
        print(f'🔄 RESUMING from: {resume_ckpt}')
    else:
        print('🆕 Starting fresh.')
else:
    print('🆕 Starting fresh.')

print(f'\n{"="*60}')
print(f'  Training {MODEL_SHORT} on {DIALECT} dialect')
print(f'  Checkpoints: {OUT_DIR}')
print(f'{"="*60}\n')

trainer.train(resume_from_checkpoint=resume_ckpt)
print('\n✅ Training complete!')

## Step 9: Save Final Model + Processor

In [ ]:
trainer.save_model(OUT_DIR)
processor.save_pretrained(OUT_DIR)
print(f'✅ Model + Processor saved to: {OUT_DIR}')

# List saved files
for item in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, item)
    if os.path.isfile(fp):
        size = os.path.getsize(fp) / (1024*1024)
        print(f'  📄 {item} ({size:.1f} MB)')
    else:
        print(f'  📁 {item}/')

## Step 10: Save Script Reference

In [ ]:
script_name = f'train_{MODEL_CHOICE}.py'
with open(os.path.join(BASE, f'scripts/{script_name}'), 'w') as f:
    f.write(f'''# {script_name} — Fine-tune {MODEL_ID} on Bangla dialect data
# Change MODEL_CHOICE and DIALECT in Phase1_Part6 notebook
MODEL_CHOICE = '{MODEL_CHOICE}'  # wav2vec2 | wavlm | hubert | data2vec
DIALECT = '{DIALECT}'            # dhaka | chittagong
# See Phase1_Part6_Train_CTC_Models.ipynb for full code
''')
print(f'✅ Script saved: scripts/{script_name}')

---
## ✅ Part 6 Complete!

### To train the next model:
1. Change `MODEL_CHOICE` in Step 1 (e.g., `'wavlm'`)
2. Re-run ALL cells from Step 1 onwards
3. Repeat for each model × dialect combination

### Minimum training runs needed:
| Run | MODEL_CHOICE | DIALECT | Priority |
|-----|-------------|---------|----------|
| 1 | `wav2vec2` | `dhaka` | **Must** |
| 2 | `wav2vec2` | `chittagong` | **Must** |
| 3 | `wavlm` | `dhaka` | **Must** |
| 4 | `wavlm` | `chittagong` | **Must** |
| 5 | `hubert` | `dhaka` | Recommended |
| 6 | `hubert` | `chittagong` | Recommended |

### Next → When all models are trained, open `Phase1_Part7_Evaluate.ipynb`

---